# Hands‑On Data Cleaning Notebook

*Generated: 2025-11-06 11:15*

This notebook guides you through a rigorous cleaning pass for your **capstone dataset**. Fill the TODOs and run cells in order. Keep this under version control (commits with meaningful messages).

## 0. Project Setup
- Make sure your raw file(s) are in `data/`.
- Update `dataset_path` below.
- Run each section and record decisions in the **Cleaning Log** at the end.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure plots appear inline (Jupyter)
# If you're running this in VS Code or other IDE, ignore the magic
%matplotlib inline

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

# Path to your raw dataset (CSV, change as needed)
dataset_path = "data/your_raw_file.csv"  # TODO: update!


## 1) Load Your Data & Initial Assessment

In [ ]:
# Load
df = pd.read_csv(dataset_path)  # If Excel: pd.read_excel(...)

print("Shape:", df.shape)
display(df.head())

# Quick statistics (numeric & datetime)
try:
    display(df.describe(include='all', datetime_is_numeric=True))
except TypeError:
    # Older pandas might not support datetime_is_numeric
    display(df.describe(include='all'))

# Sanity scan: min/max per numeric column (helps catch unbelievable values)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols:
    mins = df[numeric_cols].min(numeric_only=True)
    maxs = df[numeric_cols].max(numeric_only=True)
    display(pd.DataFrame({'min': mins, 'max': maxs}))
else:
    print("No numeric columns detected.")


## 2) Missing Data: Identify → Decide → Apply

In [ ]:
# Identify missingness
missing_counts = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing_counts / len(df) * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing_counts, "missing_pct": missing_pct})
display(missing_report)

# TODO: Decide strategy per column (drop rows, fill with mean/median/mode/specific value)
# Example template (customize carefully):
# Numerical: median; Categorical: mode
for col in df.columns:
    if df[col].isnull().any():
        if df[col].dtype.kind in "biufc":  # numeric
            fill_value = df[col].median()
            df[col] = df[col].fillna(fill_value)
        else:
            # mode() may return multiple values; take the first
            mode_vals = df[col].mode(dropna=True)
            if not mode_vals.empty:
                df[col] = df[col].fillna(mode_vals.iloc[0])

# If you prefer dropping rows with too many missing values:
# df = df.dropna(thresh=<min_non_null_per_row>)


## 3) Verify and Fix Data Types

In [ ]:
# Inspect dtypes
df.info()

# TODO: List columns that should be numeric but are 'object' (strings)
# Example coercion to numeric (invalid parsing becomes NaN, which you can then impute/handle):
# cols_to_numeric = ["colA", "colB"]  # TODO
cols_to_numeric = []  # TODO
for c in cols_to_numeric:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# TODO: Parse dates (if any)
# date_cols = ["date_col"]  # TODO
date_cols = []  # TODO
for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors="coerce")

print("\nPost-fix dtypes:")
df.info()


## 4) Remove Duplicate Records

In [ ]:
dup_count = df.duplicated().sum()
print("Duplicate rows found:", dup_count)

if dup_count > 0:
    df = df.drop_duplicates()
    print("Duplicates removed. New shape:", df.shape)
else:
    print("No duplicate rows detected.")


## 5) Detect Outliers with Visualizations

In [ ]:
# HISTOGRAMS (one plot per numeric column)
for col in numeric_cols:
    plt.figure()
    df[col].plot(kind='hist', bins=30, title=f"Histogram: {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()


In [ ]:
# BOX PLOTS (one plot per numeric column)
for col in numeric_cols:
    plt.figure()
    df.boxplot(column=[col])
    plt.title(f"Box Plot: {col}")
    plt.ylabel(col)
    plt.show()


In [ ]:
# IQR-based outlier flags (summary)
def iqr_outlier_mask(series, k=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k*iqr
    upper = q3 + k*iqr
    return (series < lower) | (series > upper)

outlier_summary = []
for col in numeric_cols:
    mask = iqr_outlier_mask(df[col].dropna())
    outlier_count = int(mask.sum())
    outlier_summary.append({"column": col, "outliers": outlier_count})
outlier_summary = pd.DataFrame(outlier_summary).sort_values("outliers", ascending=False)
display(outlier_summary)

# TODO: Investigate top outlier columns manually and decide whether to fix/remove/keep.
# Example to inspect rows:
# col_to_check = outlier_summary.iloc[0]["column"] if not outlier_summary.empty else None
# if col_to_check:
#     mask = iqr_outlier_mask(df[col_to_check])
#     display(df.loc[mask, [col_to_check]].head(20))


## 6) Save Outputs

In [ ]:
# Save cleaned dataset (CSV). Keep raw file untouched.
clean_path = "data/your_raw_file_clean.csv"  # TODO: update name
df.to_csv(clean_path, index=False)
print("Saved cleaned file to:", clean_path)

# Optional: Save a small data dictionary (columns and inferred types)
data_dict = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str),
    "non_null": df.notnull().sum(),
    "unique": df.nunique()
})
data_dict_path = "data/data_dictionary.csv"
data_dict.to_csv(data_dict_path, index=False)
print("Saved data dictionary to:", data_dict_path)


## 7) Cleaning Log (fill as you go)

**Dataset:** `data/your_raw_file.csv`  
**Team:** _Names here_  
**Date:** _YYYY‑MM‑DD_

### Decisions
- Missing values:
  - Column X → strategy (median/mode/drop), rationale: ...
  - Column Y → strategy ..., rationale: ...
- Data types:
  - Converted A → numeric; parsed B → datetime; rationale: ...
- Duplicates:
  - Found N duplicates; dropped/kept; rationale: ...
- Outliers:
  - Column Z had N outliers; we inspected and [corrected/dropped/kept] because ...

### Risks / Notes
- Any assumptions or potential data quality risks noted here.

### Next
- What else should be validated before modeling/analysis?
